In [ ]:
#Set up PySpark session with config optimized for local development
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("WildChatDataProcessing").master("local[*]").config("spark.sql.shuffle.partitions", "16").config("spark.driver.memory", "8g").config("spark.sql.parquet.mergeSchema", "false").getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/03 11:25:09 WARN Utils: Your hostname, Zipcoders-MacBook-Pro-5.local, resolves to a loopback address: 127.0.0.1; using 192.168.88.136 instead (on interface en0)
26/05/03 11:25:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 11:25:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [14]:
#Ingestion of Parquet files into a single DataFrame

paths = ["/Users/james/Projects/WildChat/data/WildChatData/0000.parquet", 
         "/Users/james/Projects/WildChat/data/WildChatData/0001.parquet", 
         "/Users/james/Projects/WildChat/data/WildChatData/0002.parquet", 
         "/Users/james/Projects/WildChat/data/WildChatData/0003.parquet", 
         "/Users/james/Projects/WildChat/data/WildChatData/0004.parquet", 
         "/Users/james/Projects/WildChat/data/WildChatData/0005.parquet", 
         "/Users/james/Projects/WildChat/data/WildChatData/0006.parquet", 
         "/Users/james/Projects/WildChat/data/WildChatData/0007.parquet", 
         "/Users/james/Projects/WildChat/data/WildChatData/0008.parquet", 
         "/Users/james/Projects/WildChat/data/WildChatData/0009.parquet", 
         "/Users/james/Projects/WildChat/data/WildChatData/0010.parquet", 
         "/Users/james/Projects/WildChat/data/WildChatData/0011.parquet", 
         "/Users/james/Projects/WildChat/data/WildChatData/0012.parquet", 
         "/Users/james/Projects/WildChat/data/WildChatData/0013.parquet"]



In [15]:
#Spot check for schema consistency across all files

schemas = {}
for path in paths:
    df = spark.read.parquet(path)
    schemas[path] = df.schema

first_schema = list(schemas.values())[0]

for path, schema in schemas.items():
    if schema != first_schema:
        print(f"Mismatch in: {path}")

In [17]:
#Basic EDA on the ingested DataFrame

parquet_files = spark.read.parquet(*paths)
parquet_files.printSchema()
parquet_files.count()
len(parquet_files.columns)
parquet_files.select(parquet_files.columns[:10]).show(5, truncate=False)
parquet_files.write.mode("overwrite").parquet("/Users/james/Projects/WildChat/data/WildChatData/ingested_wild_chat_data.parquet")

root
 |-- conversation_hash: string (nullable = true)
 |-- model: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- conversation: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- content: string (nullable = true)
 |    |    |-- country: string (nullable = true)
 |    |    |-- hashed_ip: string (nullable = true)
 |    |    |-- header: struct (nullable = true)
 |    |    |    |-- accept-language: string (nullable = true)
 |    |    |    |-- user-agent: string (nullable = true)
 |    |    |-- language: string (nullable = true)
 |    |    |-- redacted: boolean (nullable = true)
 |    |    |-- role: string (nullable = true)
 |    |    |-- state: string (nullable = true)
 |    |    |-- timestamp: timestamp (nullable = true)
 |    |    |-- toxic: boolean (nullable = true)
 |    |    |-- turn_identifier: long (nullable = true)
 |-- turn: long (nullable = true)
 |-- language: string (nullable = true)
 |-- openai_moderation: array (nu

In [ ]:
#Global temp view created to enable SQL queries across the ingested data

parquet_files.createOrReplaceGlobalTempView("wild_chat_data")